# 09 - Scenario and Model Comparison

One fixed test set. Every training set that matters, and every classifier.

**Three scenarios per generative model:**

| Scenario | Training set | Question it answers |
|---|---|---|
| 1 | `df_train.csv` - original only | What does the data give without any augmentation? |
| 2 | `<stem>.csv` - original + synthetic | Does augmentation add anything? |
| 3 | `<stem>_synthetic_only.csv` - synthetic only | Did the generator learn the real structure at all? |

Scenario 1 is evaluated once and serves as the reference for every comparison.

This notebook replaces the earlier `08b` and `09`, which measured overlapping
things. What it reports:

1. Accuracy, macro F1 and macro ROC-AUC for every training set and classifier.
2. **McNemar's test** against scenario 1. With 13,000 test records the standard
   error of accuracy is about 0.26 percentage points, so a raw gap of a few
   tenths means nothing on its own.
3. **Per-class recall**, from the scarcest class upward.
4. An **accuracy cross-check** against the confusion matrix, and a **separation
   audit** of every training set against the test set.

The hyperparameter search runs **once per classifier**, on scenario 1. Searching
per training set would confound the data with the tuning.


## 1. Upload

In [ ]:
# Upload df_train.csv, df_test.csv, and for each generative model and
# epoch setting you want to compare, both files written by notebooks 02-05:
#     <stem>.csv                 and     <stem>_synthetic_only.csv
from google.colab import files

uploaded = files.upload()
for name in sorted(uploaded):
    print(name)


## 2. Imports, seeds and switches

In [ ]:
import os, json, random, platform
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             recall_score, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import binomtest

import lightgbm as lgb
import xgboost as xgb

SEED = 42
N_SPLITS = 5
SHUFFLE = True
TARGET_COL = 'Target'

random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

USE_GPU = True
XGB_DEVICE_KW = {}
if USE_GPU:
    try:
        if int(xgb.__version__.split('.')[0]) >= 2:
            XGB_DEVICE_KW = {'device': 'cuda', 'tree_method': 'hist'}
        else:
            XGB_DEVICE_KW = {'tree_method': 'gpu_hist'}
    except Exception as e:
        print('GPU unavailable, using CPU:', e)

PARAMS_FILE = 'best_params_09.json'
TUNE = True            # True on the first run, False afterwards

print('=== REPRODUCIBILITY ===')
print('Global seed      :', SEED)
print('Cross-validation : StratifiedKFold(n_splits={}, shuffle={}, random_state={})'
      .format(N_SPLITS, SHUFFLE, SEED))
print('Search           : RandomizedSearchCV(n_iter=5, scoring=f1_macro, random_state={})'
      .format(SEED))
print('Generation seed  : {} (notebooks 02-05)'.format(SEED))
print('Python           :', platform.python_version())
print('XGBoost device   :', XGB_DEVICE_KW or 'CPU')


## 3. Which runs to compare

Each entry expands into up to two training sets - the combined file and the
synthetic-only file. Files that were not uploaded are skipped with a note, so
you can start with one model and add the rest later.

Every fit costs time. 12 runs x 2 scenarios x 3 classifiers is 72 fits plus the
baseline. Start small.


In [ ]:
DEVICE_TAG = 'gpu'
TARGET_PER_CLASS = 6000

# (model tag, epochs) - the model tag must match what notebooks 02-05 wrote
RUNS = [
    ('ctgan_perclass', 100), ('ctgan_perclass', 300), ('ctgan_perclass', 500),
    ('cwgan', 100),          ('cwgan', 300),          ('cwgan', 500),
    ('tablegan', 100),       ('tablegan', 300),       ('tablegan', 500),
    ('medgan', 100),         ('medgan', 300),         ('medgan', 500),
]

INCLUDE_COMBINED = True     # scenario 2
INCLUDE_SYNTHETIC = True    # scenario 3
CLASSIFIERS = ['lightgbm', 'xgboost', 'random_forest']

BASELINE_LABEL = '1. Baseline (no augmentation)'

# ---- build the list of training sets ---------------------------
training_sets = [(BASELINE_LABEL, 'df_train.csv', 1)]
missing = []

for model, epochs in RUNS:
    stem = f'{model}_{DEVICE_TAG}_epoch{epochs}_augment{TARGET_PER_CLASS}_seed{SEED}'
    pretty = f'{model.replace("_perclass", "").upper()} {epochs}'

    if INCLUDE_COMBINED:
        path = f'{stem}.csv'
        (training_sets if os.path.exists(path) else missing).append(
            (f'2. {pretty} combined', path, 2) if os.path.exists(path) else path)

    if INCLUDE_SYNTHETIC:
        path = f'{stem}_synthetic_only.csv'
        (training_sets if os.path.exists(path) else missing).append(
            (f'3. {pretty} synthetic', path, 3) if os.path.exists(path) else path)

df_test = pd.read_csv('df_test.csv')
X_test = df_test.drop(columns=TARGET_COL)
y_test = df_test[TARGET_COL].to_numpy()
FEATURES = X_test.columns.tolist()

print('Test set :', df_test.shape, '| classes:', np.bincount(y_test).tolist())

# Optional third partition, written by notebook 01 when
# UNSEEN_PER_CLASS > 0. Raw records that never went through the
# outlier and zero filtering, so they represent what arrives in
# practice rather than what survives cleaning.
UNSEEN_FILE = 'df_unseen.csv'

if os.path.exists(UNSEEN_FILE):
    df_unseen = pd.read_csv(UNSEEN_FILE)
    X_unseen = df_unseen[FEATURES]
    y_unseen = df_unseen[TARGET_COL].to_numpy()
    print('Unseen   :', df_unseen.shape, '| classes:', np.bincount(y_unseen).tolist())

    _out = int((X_unseen < 0).sum().sum() + (X_unseen > 1).sum().sum())
    print('           {} values outside [0, 1] - expected, it was never cleaned'
          .format(_out))
else:
    df_unseen = X_unseen = y_unseen = None
    print('Unseen   : df_unseen.csv not uploaded, that evaluation is skipped')
print('\nTraining sets found:', len(training_sets))
for label, path, scen in training_sets:
    print('   [{}] {:<30} {}'.format(scen, label, path))

if missing:
    print('\nNot uploaded, skipped:', len(missing))
    for path in missing:
        print('   ', path)

print('\nTotal fits to run:', len(training_sets) * len(CLASSIFIERS))


## 4. Separation audit

Every training set is checked against the test set before anything is trained.
A non-zero count for a scenario 2 or 3 file would mean a generator reproduced a
held-out record - the leakage the reviewer asked about.


In [ ]:
test_keys = set(map(tuple, np.round(df_test[FEATURES].to_numpy(dtype=float), 6)))

rows = []
for label, path, scen in training_sets:
    df = pd.read_csv(path)
    keys = set(map(tuple, np.round(df[FEATURES].to_numpy(dtype=float), 6)))
    counts = df[TARGET_COL].value_counts().sort_index()
    rows.append({'Scenario': scen, 'Training set': label, 'Rows': len(df),
                 'Min/class': int(counts.min()), 'Max/class': int(counts.max()),
                 'Shared with test': len(keys & test_keys)})

audit = pd.DataFrame(rows)
display(audit)

if audit['Shared with test'].sum() == 0:
    print('No training set shares a feature vector with the test set.')
else:
    print('WARNING - overlap detected. Investigate before reporting anything.')


## 5. Estimators, search spaces, and the tune-once helper

In [ ]:
SPACES = {
    'lightgbm': {
        'n_estimators': [100, 150, 200], 'learning_rate': [0.05, 0.1],
        'num_leaves': [31, 64, 90], 'max_depth': [3, 5, 10, -1],
        'min_child_samples': [10, 20, 40],
        'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0]},
    'xgboost': {
        'n_estimators': [100, 150, 200], 'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5, 10, 0], 'min_child_weight': [1, 3, 5],
        'gamma': [0, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0]},
    'random_forest': {
        'n_estimators': [100, 150, 200], 'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]},
}


def make_estimator(name, params=None):
    params = dict(params or {})
    if name == 'lightgbm':
        return lgb.LGBMClassifier(objective='multiclass', random_state=SEED,
                                  verbosity=-1, **params)
    if name == 'xgboost':
        return xgb.XGBClassifier(objective='multi:softprob', random_state=SEED,
                                 verbosity=0, eval_metric='mlogloss',
                                 **XGB_DEVICE_KW, **params)
    return RandomForestClassifier(random_state=SEED, n_jobs=-1, **params)


def _store():
    if os.path.exists(PARAMS_FILE):
        with open(PARAMS_FILE) as fh:
            return json.load(fh)
    return {}


def tuned_params(name, X, y):
    """Search once on the baseline; reuse for every other training set."""
    store = _store()
    if not TUNE and name in store:
        print('  reusing stored hyperparameters')
        return dict(store[name])

    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=SHUFFLE, random_state=SEED)
    search = RandomizedSearchCV(make_estimator(name), SPACES[name], n_iter=5,
                                scoring='f1_macro', cv=kf, random_state=SEED,
                                n_jobs=-1, verbose=0)
    search.fit(X, y)
    params = dict(search.best_params_)
    print('  CV macro F1 : {:.4f}'.format(search.best_score_))
    print('  params      :', params)

    store[name] = params
    with open(PARAMS_FILE, 'w') as fh:
        json.dump(store, fh, indent=2)
    return params


print('Ready. Classifiers:', CLASSIFIERS)


## 6. Run everything

In [ ]:
def load_training(path):
    df = pd.read_csv(path)
    return df[FEATURES], df[TARGET_COL].to_numpy()


X_base, y_base = load_training(training_sets[0][1])

results, predictions, unseen_predictions = [], {}, {}

for name in CLASSIFIERS:
    print('\n' + '=' * 70)
    print('CLASSIFIER:', name)
    print('=' * 70)
    print('Tuning on', BASELINE_LABEL)
    params = tuned_params(name, X_base, y_base)

    for label, path, scen in training_sets:
        X_tr, y_tr = load_training(path)

        model = make_estimator(name, params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)
        cm = confusion_matrix(y_test, y_pred)

        acc = accuracy_score(y_test, y_pred)
        assert abs(acc - np.trace(cm) / cm.sum()) < 1e-9, \
            'confusion matrix disagrees with accuracy_score'

        row = {
            'Classifier': name, 'Scenario': scen, 'Training set': label,
            'Train rows': len(X_tr), 'Accuracy': acc,
            'Macro F1': f1_score(y_test, y_pred, average='macro'),
            'Macro ROC-AUC': roc_auc_score(y_test, y_prob, multi_class='ovr',
                                           average='macro'),
            'Correct': int(np.trace(cm)), 'Total': int(cm.sum()),
        }

        # The same fitted model, scored on the raw unseen partition
        if X_unseen is not None:
            u_pred = model.predict(X_unseen)
            row['Unseen accuracy'] = accuracy_score(y_unseen, u_pred)
            row['Unseen macro F1'] = f1_score(y_unseen, u_pred, average='macro')
            unseen_predictions[(name, label)] = u_pred

        results.append(row)
        predictions[(name, label)] = y_pred

        line = ('  [{}] {:<30} test acc {:.4f} | F1 {:.4f} | {}/{}'
                .format(scen, label, acc, row['Macro F1'],
                        row['Correct'], row['Total']))
        if X_unseen is not None:
            line += ' | unseen acc {:.4f}'.format(row['Unseen accuracy'])
        print(line)

df_results = pd.DataFrame(results)
print('\n' + '=' * 70)
display(df_results.round(4))


## 7. Summary and accuracy cross-check

The `Correct` and `Total` columns come straight from the confusion matrix, so a
figure and this table drawn from the same run cannot disagree - which is what
the reviewer caught between Figure 3a and Table 9.


In [ ]:
check = df_results.copy()
check['From matrix'] = check['Correct'] / check['Total']
check['Difference'] = (check['Accuracy'] - check['From matrix']).abs()
print('Largest accuracy discrepancy:', check['Difference'].max())
assert check['Difference'].max() < 1e-9
print('Accuracy and confusion matrix agree for every run.\n')

pivot = df_results.pivot_table(index='Training set', columns='Classifier',
                               values='Macro F1').round(4)
order = [t[0] for t in training_sets]
display(pivot.reindex(order))

print('\nBest overall:')
best = df_results.loc[df_results['Macro F1'].idxmax()]
print('  {} | {} | F1 {:.4f} | acc {:.4f}'
      .format(best['Training set'], best['Classifier'],
              best['Macro F1'], best['Accuracy']))

base_f1 = df_results[df_results['Training set'] == BASELINE_LABEL]['Macro F1'].max()
n_better = (df_results[df_results['Scenario'] == 2]['Macro F1'] > base_f1).sum()
n_total = (df_results['Scenario'] == 2).sum()
print('  Augmented runs beating the best baseline: {} of {}'.format(n_better, n_total))


## 7b. Curated conditions versus realistic conditions

`df_test` is balanced and cleaned: outlier rows and rows holding a zero were
removed with the training-set rule. `df_unseen` went through none of that, so it
still holds the extreme values and the zero-coded missing entries that arrive in
practice.

The gap between the two is the result. It measures how much of the reported
performance depends on the cleaning rather than on the model.

At 1,300 records the 95% margin of error on the unseen accuracy is roughly
+/- 1.6 percentage points, wider than the spread between configurations. Report
one honest headline number from it; keep the ranking on `df_test`.


In [ ]:
if X_unseen is None:
    print('df_unseen.csv was not uploaded, nothing to compare.')
else:
    gap = df_results[['Classifier', 'Scenario', 'Training set',
                      'Accuracy', 'Unseen accuracy',
                      'Macro F1', 'Unseen macro F1']].copy()
    gap['Accuracy drop'] = gap['Accuracy'] - gap['Unseen accuracy']
    gap['F1 drop'] = gap['Macro F1'] - gap['Unseen macro F1']
    gap = gap.sort_values('Accuracy drop')

    display(gap.round(4))

    print('\nAccuracy drop from curated to realistic conditions:')
    print('  smallest : {:+.4f}  ({})'.format(
        gap['Accuracy drop'].iloc[0], gap['Training set'].iloc[0]))
    print('  largest  : {:+.4f}  ({})'.format(
        gap['Accuracy drop'].iloc[-1], gap['Training set'].iloc[-1]))
    print('  mean     : {:+.4f}'.format(gap['Accuracy drop'].mean()))

    if gap['Accuracy drop'].mean() > 0.02:
        print('\nThe drop is substantial. State it in the Limitations: the')
        print('headline figures hold for cleaned records, not for raw ones.')
    else:
        print('\nThe drop is small, which is itself worth reporting: the')
        print('model degrades gracefully on records that were never filtered.')


## 8. McNemar against the baseline

* **b** - the baseline got it right, this training set got it wrong
* **c** - the baseline got it wrong, this training set got it right

Under the null hypothesis b follows Binomial(b + c, 0.5), tested exactly.
`p >= 0.05` means no significant difference - a stronger and more defensible
claim for the paper than saying augmentation made things worse.


In [ ]:
rows = []
for name in CLASSIFIERS:
    base = predictions.get((name, BASELINE_LABEL))
    if base is None:
        continue
    base_ok = (base == y_test)

    for label, path, scen in training_sets[1:]:
        other = predictions.get((name, label))
        if other is None:
            continue
        other_ok = (other == y_test)

        b = int(np.sum(base_ok & ~other_ok))
        c = int(np.sum(~base_ok & other_ok))
        p = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue

        if p >= 0.05:
            verdict = 'no significant difference'
        elif c > b:
            verdict = 'significantly BETTER'
        else:
            verdict = 'significantly WORSE'

        rows.append({'Classifier': name, 'Scenario': scen, 'Training set': label,
                     'b': b, 'c': c, 'Net': c - b, 'p-value': p,
                     'Verdict at 0.05': verdict})

df_mcnemar = pd.DataFrame(rows)
display(df_mcnemar.round(6))

if len(df_mcnemar):
    print('\nVerdict counts:')
    print(df_mcnemar['Verdict at 0.05'].value_counts().to_string())


## 9. Per-class recall

Sorted from the scarcest class in the original training set. Augmentation may
lift the rare classes while costing the common ones; a single accuracy figure
hides that, this table does not.


In [ ]:
classes = np.unique(y_test)
base_counts = np.bincount(pd.read_csv(training_sets[0][1])[TARGET_COL].to_numpy())

recall_table = pd.DataFrame({'Class': classes,
                             'Train n (baseline)': base_counts[classes]})

for (name, label), y_pred in predictions.items():
    recall_table['{} | {}'.format(name, label)] = recall_score(
        y_test, y_pred, average=None, labels=classes)

recall_table = recall_table.sort_values('Train n (baseline)').reset_index(drop=True)
display(recall_table.round(4))

perfect = [int(c) for c in classes
           if all(recall_table.loc[recall_table['Class'] == c, col].iloc[0] == 1.0
                  for col in recall_table.columns[2:])]
if perfect:
    print('\nClasses at recall 1.0 in EVERY training set, baseline included:', perfect)
    print('A class that is perfect without any augmentation cannot owe that')
    print('result to the generator. This is the evidence for the reviewer.')


## 10. Save

In [ ]:
df_results.to_csv('09_results.csv', index=False)
recall_table.to_csv('09_per_class_recall.csv', index=False)
audit.to_csv('09_separation_audit.csv', index=False)
if len(df_mcnemar):
    df_mcnemar.to_csv('09_mcnemar.csv', index=False)

pd.DataFrame({'{} | {}'.format(n, l): p for (n, l), p in predictions.items()}) \
  .assign(y_true=y_test).to_csv('09_predictions.csv', index=False)

if unseen_predictions:
    pd.DataFrame({'{} | {}'.format(n, l): p
                  for (n, l), p in unseen_predictions.items()}) \
      .assign(y_true=y_unseen).to_csv('09_unseen_predictions.csv', index=False)
    gap.to_csv('09_curated_vs_realistic.csv', index=False)

for f in ['09_results.csv', '09_per_class_recall.csv', '09_separation_audit.csv',
          '09_mcnemar.csv', '09_predictions.csv',
          '09_unseen_predictions.csv', '09_curated_vs_realistic.csv']:
    if os.path.exists(f):
        print('saved', f)


## Reading the result

**Scenario 3 first.** If synthetic data alone trains a classifier close to the
baseline, the generator learned the real structure, and scenario 2 sitting level
with scenario 1 simply means there was no headroom left. If scenario 3
collapses, the synthetic data does not carry the signal and scenario 2 works
only because the original rows inside it do the work.

**Then scenario 2 against scenario 1, with McNemar.** That is what settles
whether augmentation helped.

| Outcome | What to write |
|---|---|
| Scenario 2 significantly better | Augmentation helps; report where, and by how much |
| No significant difference | Even with valid synthetic values, augmentation brings no benefit in this regime |
| Scenario 2 significantly worse | Synthetic data is harmful here; the mode collapse in the KDE plots is the explanation |

The per-class recall table is where the contribution lives in the second and
third cases: it shows *where* augmentation moved performance, even when the
total did not move.
